## Portfolio - Predicción de goles en partidos FIFA WC26

### Captura de datos (Webscrapping)

En la construcción de un modelo de predicción de goles, es necesario obtener datos. Para ello, se van a webscrapear los resultados que se encuentran en Wikipedia de las 48 selecciones que participan en la FIFA WC 26

### Librerias utilizadas

- Pandas
- Beautiful Soup
- requests
- re (Regular Expressions).
- utils.py (archivo con las funciones a utilizar para webscrapear)

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import requests
import re
import utils as utils

### 1 - Captura del Ranking FIFA de selecciones

Mediante una API de FIFA se obtiene el ranking de selecciones. 

De aquí vamos a capturar los códigos de país y el Score FIFA 

In [2]:
if __name__ == "__main__":
    url_tu_api = "https://api.fifa.com/api/v3/fifarankings/rankings/live?gender=1&sportType=0&language=en" 
    df_ranking_fifa = utils.obtener_ranking_fifa_oficial(url_tu_api)
    
    if not df_ranking_fifa.empty:
        print("\n--- VISTA PREVIA DEL RANKING FIFA OBTENIDO ---")
        print(df_ranking_fifa.sample(5).to_string(index=False))

Conectando y procesando el JSON de la FIFA...
¡Éxito! Se importaron 211 selecciones desde el Ranking FIFA.

--- VISTA PREVIA DEL RANKING FIFA OBTENIDO ---
 Ranking_Actual       Seleccion  Puntos_FIFA Codigo_Pais
             54         Tunisia  1453.004172         TUN
             23  Korea Republic  1591.749573         KOR
            164            Cuba   981.423455         CUB
            125        Suriname  1132.425895         SUR
            153 Solomon Islands  1031.894156         SOL


### 2 - Captura Webscrapping

Listado paises que participan en la FIFA WC26 en Wikipedia

In [3]:
url_mundial = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    respuesta = requests.get(url_mundial, headers=headers)
    tablas = pd.read_html(respuesta.text)
except Exception as e:
    print(f"Error al leer las tablas del Mundial: {e}")

patron = r'([A-Za-z\s]+?)(?:\[.*?\])?\s*\((\d+)\)'
confederaciones_ignorar = ['AFC', 'CAF', 'CONCACAF', 'CONMEBOL', 'OFC', 'UEFA']

datos_procesados = []

for columna in tablas[5].columns:
    texto_celda = str(tablas[5][columna].iloc[0])
    # Aplicamos la expresión regular para buscar todos los países y rankings en esa celda
    coincidencias = re.findall(patron, texto_celda)
    # Limpiamos y guardamos los datos encontrados
    for pais, ranking in coincidencias:
        pais_limpio = pais.strip()
        if pais_limpio not in confederaciones_ignorar:
            # Aquí obtenemos el código automáticamente del diccionario
            codigo = utils.obtener_codigo_pais(pais_limpio, df_ranking_fifa)
            datos_procesados.append({
                "Seleccion": pais_limpio,
                "Codigo_Pais": codigo,
                "Ranking_FIFA": int(ranking)
            })

# 4. Convertimos la lista de todos los países encontrados en un DataFrame final
ranking_teams = pd.DataFrame(datos_procesados)
ranking_teams.replace({'ao': 'Curacao'}, inplace=True)
print('********* Q Teams:', len(ranking_teams), '-*********')
print(ranking_teams.sample(5).to_string(index=False))

********* Q Teams: 48 -*********
Seleccion Codigo_Pais  Ranking_FIFA
    Qatar         QAT            56
   Brazil         BRA             6
  Ecuador         ECU            23
  Austria         AUT            24
    Spain         ESP             2


### 3 - Captura Webscrapping

Captura de partidos anteriores por selección, posterior al mundial de Catar 2022.
Se encontraron 5 formatos distintos de Website en Wikipedia, añadiendo las funciones de Webscrapping en utils.py

Captura BBDD según código país y tipo pagina wikipedia

In [4]:
lst_1 = ['ARG', 'COL', 'BRA', 'ECU', 'PAR', 'HAI', 'NZL', 'SCO']
lst_2 = ['URU', 'AUS', 'BEL', 'IRQ', 'JPN', 'JOR', 'QAT', 'KSA', 'KOR', 'UZB',
        'ALG', 'CPV', 'COD', 'EGY', 'GHA', 'CIV', 'MAR', 'SEN', 'RSA', 'TUN',
        'CUW', 'MEX', 'PAN', 'AUT', 'BIH', 'CRO', 'CZE', 'ENG', 'FRA', 
        'GER', 'NOR', 'POR', 'ESP', 'SWE', 'SUI', 'TUR']
lst_3 = ['USA', 'CAN']
lst_4 = ['NED']
lst_5 = ['IRN']

In [5]:
df_games = pd.DataFrame()

for lst in lst_1:
    _df = utils.games_type01(utils.records_teams[lst], lst)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_2:
    _df = utils.games_type02(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_3:
    _df = utils.games_type03(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])
    
for lst in lst_4:
    _df = utils.games_type04(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])    
    
for lst in lst_5:
    _df = utils.games_iran(utils.records_teams[lst], lst, df_ranking_fifa)
    df_games = pd.concat([df_games, _df])                

Iniciando extracción de datos para: ARG...
Iniciando extracción de datos para: COL...
Iniciando extracción de datos para: BRA...
Iniciando extracción de datos para: ECU...
Iniciando extracción de datos para: PAR...
Iniciando extracción de datos para: HAI...
Iniciando extracción de datos para: NZL...
Iniciando extracción de datos para: SCO...
Iniciando extracción robusta para: URU...
Iniciando extracción robusta para: AUS...
Iniciando extracción robusta para: BEL...
Iniciando extracción robusta para: IRQ...
Iniciando extracción robusta para: JPN...
Iniciando extracción robusta para: JOR...
Iniciando extracción robusta para: QAT...
Iniciando extracción robusta para: KSA...
Iniciando extracción robusta para: KOR...
Iniciando extracción robusta para: UZB...
Iniciando extracción robusta para: ALG...
Iniciando extracción robusta para: CPV...
Iniciando extracción robusta para: COD...
Iniciando extracción robusta para: EGY...
Iniciando extracción robusta para: GHA...
Iniciando extracción robus

Se agrega un archivo con Partidos cque no se encuentran en Wikipedia (aprox 50 partidos)

In [12]:
df_lost = pd.read_excel('../database/OtherGames.xlsx').drop(columns=['Codigo_Pais', 'Tipo_Partido_Limpio'])
df_lost['Fecha'] = df_lost['Fecha'].dt.strftime("%Y-%m-%d")
df_final = pd.concat([df_games, df_lost], axis=0)
df_final.rename(columns={'Seleccion':'Team', 
        'Condicion':'Condition', 'Goles_Equipo':'Goals_team', 
        'Goles_Rival':'Goals_opponent', 'Es_Penales':'Penalties', 
        'Es_Tiempo_Extra':'AET', 'Fecha':'Date', 
        'Tipo_Partido':'Game_type', 'Oponente':'Opponent'}, inplace=True)
df_final = df_final.reset_index(drop=True)

Save DataFrames

In [14]:
df_final.to_parquet('../database/games.parquet', index=False)
df_ranking_fifa.to_parquet('../database/ranking_fifa.parquet', index=False)
ranking_teams.to_parquet('../database/teams.parquet', index=False)